In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm 
import pyblp

import warnings
warnings.filterwarnings('ignore')

Question 1

In [3]:
df = pd.read_csv('pset2_data.csv', index_col=0)

inside_share = df.groupby('market_ids')['shares'].sum()
df = df.merge(inside_share.rename('inside_total'), on='market_ids')
df['s0'] = 1-df['inside_total']

df['y'] = np.log(df['shares'])-np.log(df['s0'])

X = df[['x','satellite', 'wired', 'prices']]
y = df['y']

model = sm.OLS(y,X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.160
Model:                            OLS   Adj. R-squared:                  0.159
Method:                 Least Squares   F-statistic:                     152.5
Date:                Fri, 17 Apr 2026   Prob (F-statistic):           1.91e-90
Time:                        20:16:52   Log-Likelihood:                -4702.0
No. Observations:                2400   AIC:                             9412.
Df Residuals:                    2396   BIC:                             9435.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
x              0.9419      0.122      7.752      0.0

Question 2

In [4]:
from linearmodels.iv import IV2SLS

model = IV2SLS(dependent=df['y'], exog=  df[['x','satellite', 'wired']], endog = df[['prices']], instruments =df[['w']]).fit()

print(model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                      y   R-squared:                     -0.0841
Estimator:                    IV-2SLS   Adj. R-squared:                -0.0855
No. Observations:                2400   F-statistic:                    764.80
Date:                Fri, Apr 17 2026   P-value (F-stat)                0.0000
Time:                        20:16:52   Distribution:                  chi2(4)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
x              1.0677     0.1455     7.3375     0.0000      0.7825      1.3529
satellite      4.7245     1.0547     4.4793     0.00

Question 3

In [5]:
df['nest'] = df['satellite'].astype(int)

group_share = df.groupby(['market_ids', 'nest'])['shares'].sum()
df = df.merge(group_share.rename('nest_share'), on=['market_ids', 'nest'])
df['within_share'] = df['shares'] / df['nest_share']
df['log_within_share'] = np.log(df['within_share'])

df['rho_sat'] = df['satellite'] * df['log_within_share']
df['rho_wired'] = df['wired'] * df['log_within_share']
 

df['other_x_in_nest'] = df.groupby(['market_ids', 'nest'])['x'].transform('sum') - df['x']
df['other_w_in_nest'] = df.groupby(['market_ids', 'nest'])['w'].transform('sum') - df['w']
 
total_x = df.groupby('market_ids')['x'].transform('sum')
nest_x = df.groupby(['market_ids', 'nest'])['x'].transform('sum')
df['other_nest_x'] = total_x - nest_x
 
total_w = df.groupby('market_ids')['w'].transform('sum')
nest_w = df.groupby(['market_ids', 'nest'])['w'].transform('sum')
df['other_nest_w'] = total_w - nest_w
 
df['sat_other_x'] = df['satellite'] * df['other_x_in_nest']
df['sat_other_w'] = df['satellite'] * df['other_w_in_nest']


model = IV2SLS(dependent=df['y'], exog=  df[['x','satellite', 'wired']], endog = df[['prices', 'rho_sat', 'rho_wired']], instruments =df[['w', 'other_x_in_nest',
        'other_w_in_nest', 'other_nest_x', 'other_nest_w', 'sat_other_x', 'sat_other_w']]).fit()

print(model.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                      y   R-squared:                      0.2105
Estimator:                    IV-2SLS   Adj. R-squared:                 0.2089
No. Observations:                2400   F-statistic:                    1063.4
Date:                Fri, Apr 17 2026   P-value (F-stat)                0.0000
Time:                        20:16:53   Distribution:                  chi2(6)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
x              0.8848     0.1630     5.4287     0.0000      0.5654      1.2043
satellite      4.1862     0.9141     4.5796     0.00

Question 4

In [6]:
alpha = -1.9937
beta1 = 0.8848
beta2 = 4.1862
beta3 = 4.1951
rho = {1: 0.2761, 0: 0.2979}

T = 600
J = 4

df['xi'] = df['y'] - (beta1*df['x']+beta2*df['satellite']+ beta3*df['wired'] + alpha*df['prices']+ rho[1]*df['satellite'] * np.log(df['within_share']) + rho[0]*df['wired'] * np.log(df['within_share']))
df['delta'] = (beta1*df['x']+ beta2*df['satellite'] + beta3*df['wired'] + alpha*df['prices'] + df['xi'])


def nested_logit_shares(deltas, nests, sigmas):
    J = len(deltas)
    unique_nests = np.unique(nests)
    nest_iv = {}
    for g in unique_nests:
        members = np.where(nests == g)[0]
        nest_iv[g] = np.sum(np.exp(deltas[members] / (1-sigmas[g])))
    denom= 1.0
    for g in unique_nests:
        denom += nest_iv[g]**(1-sigmas[g])
    shares = np.zeros(J)
    for j in range(J):
        g = nests[j]
        s_jg = np.exp(deltas[j]/(1-sigmas[g])) / nest_iv[g]
        s_g = nest_iv[g] ** (1-sigmas[g]) /denom
        shares[j] = s_jg*s_g
    return shares


sigmas = {1: rho[1], 0: rho[0]}
eps = 1e-6
avg_elast = np.zeros((J,J))
avg_diversion = np.zeros((J,J))

for t in range(T):
    mkt = df[df['market_ids']==t].sort_values('firm_ids')
    deltas = mkt['delta'].values.copy()
    p =mkt['prices'].values.copy()
    nests_t = mkt['nest'].values

    s0 = nested_logit_shares(deltas, nests_t, sigmas)
    for k in range(J):
        deltas_up = deltas.copy()
        deltas_up[k] += alpha*eps
        s1= nested_logit_shares(deltas_up, nests_t, sigmas)
        
        for j in range(J):
            avg_elast[j,k] += (s1[j]-s0[j]) /eps*p[k]/s0[j]
    for j in range(J):
        deltas_up = deltas.copy()
        deltas_up[j] += alpha*eps
        s1= nested_logit_shares(deltas_up, nests_t, sigmas)
        ds_j = s1[j]-s0[j]
        for k in range(J):
            if j != k:
                avg_diversion[j,k] += (s1[k]-s0[k]) / (-ds_j)

avg_elast /= T
avg_diversion /= T

true_elast = pd.read_csv('true_ownprice_elasticities.csv', index_col=0)
true_div = pd.read_csv('true_diversionratio.csv', index_col=0)


print("OWN-PRICE ELASTICITIES")
print(f"{'Product':<10} {'Estimated':<15} {'True':<15}")
for j in range(J):
    print(f"J{j+1:<9} {avg_elast[j,j]:<15.4f} {true_elast.iloc[0, j]:<15.4f}")

print("\nESTIMATED DIVERSION RATIOS")
print(pd.DataFrame(avg_diversion, index=['J1','J2','J3','J4'], columns=['J1','J2','J3','J4']).round(4))

print("\nTRUE DIVERSION RATIOS")
print(true_div.round(4))

    

OWN-PRICE ELASTICITIES
Product    Estimated       True           
J1         -5.3543         -4.2500        
J2         -5.5505         -4.3714        
J3         -5.5186         -4.2711        
J4         -5.5122         -4.2576        

ESTIMATED DIVERSION RATIOS
        J1      J2      J3      J4
J1  0.0000  0.3334  0.1865  0.1943
J2  0.3386  0.0000  0.1856  0.1861
J3  0.1839  0.1827  0.0000  0.3526
J4  0.1868  0.1771  0.3508  0.0000

TRUE DIVERSION RATIOS
        J1      J2      J3      J4
J1  0.3338  0.2170  0.2204  0.2288
J2  0.2194  0.3370  0.2216  0.2220
J3  0.2193  0.2185  0.3350  0.2272
J4  0.2246  0.2150  0.2232  0.3371


Question 5

In [16]:
pyblp.options.digits = 4
pyblp.options.verbose = True
pyblp.options.verbose = False

df['product_ids'] = df['firm_ids']
product_data = df[['market_ids', 'firm_ids', 'product_ids', 'shares', 'prices', 'x', 'satellite', 'wired', 'w']].copy()


product_data['nest'] = product_data['satellite'].astype(int)
product_data['total_x'] = product_data.groupby('market_ids')['x'].transform('sum')
product_data['total_w'] = product_data.groupby('market_ids')['w'].transform('sum')
product_data['nest_x'] = product_data.groupby(['market_ids', 'nest'])['x'].transform('sum')
product_data['nest_w'] = product_data.groupby(['market_ids', 'nest'])['w'].transform('sum')

product_data['demand_instruments0'] = product_data['w']
product_data['demand_instruments1'] = product_data['nest_x'] - product_data['x']     
product_data['demand_instruments2'] = product_data['total_x'] - product_data['nest_x']  
product_data['demand_instruments3'] = product_data['nest_w'] - product_data['w']       
product_data['demand_instruments4'] = product_data['total_w'] - product_data['nest_w']  

product_data['supply_instruments0'] = product_data['x']
product_data['supply_instruments1'] = product_data['nest_x'] - product_data['x']
product_data['supply_instruments2'] = product_data['total_x'] - product_data['nest_x']
product_data['supply_instruments3'] = product_data['nest_w'] - product_data['w']
product_data['supply_instruments4'] = product_data['total_w'] - product_data['nest_w']

integration = pyblp.Integration('product', size=7)

formulation1 = pyblp.Formulation('0 + x + satellite + wired + prices')  
formulation2 = pyblp.Formulation('0 + satellite + wired')               
formulation3 = pyblp.Formulation('1 + w')                             
 
initial_sigma = np.eye(2)  


print("=" * 60)
print("SPEC 1: Demand only, BLP instruments")
print("=" * 60)
 
problem1 = pyblp.Problem(product_formulations=(formulation1, formulation2), product_data=product_data, integration=integration)
print(problem1)
 
results1 = problem1.solve(initial_sigma, optimization=pyblp.Optimization('l-bfgs-b'), iteration=pyblp.Iteration('squarem', {'atol': 1e-14}))
print(results1)

print("\n" + "=" * 60)
print("SPEC 2: Demand + Supply, BLP instruments")
print("=" * 60)
 
problem2 = pyblp.Problem(product_formulations=(formulation1, formulation2, formulation3),product_data=product_data, integration=integration, costs_type='log')
print(problem2)
 
results2 = problem2.solve(initial_sigma, beta=[None, None, None, -2.0], optimization=pyblp.Optimization('l-bfgs-b'), iteration=pyblp.Iteration('squarem', {'atol': 1e-14}))
print(results2)


print("\n" + "=" * 60)
print("SPEC 3: Demand + Supply, Optimal instruments")
print("=" * 60)

optimal_instruments = results2.compute_optimal_instruments()
optimal_problem = optimal_instruments.to_problem()
print(optimal_problem)

results3 = optimal_problem.solve(results2.sigma, beta=[None, None, None, results2.beta[3, 0]], optimization=pyblp.Optimization('l-bfgs-b', {'maxfun': 500}), iteration=pyblp.Iteration('squarem', {'atol': 1e-14}), method='1s')


results3 = optimal_problem.solve(results3.sigma, beta=[None, None, None, results3.beta[3, 0]], optimization=pyblp.Optimization('l-bfgs-b'), iteration=pyblp.Iteration('squarem', {'atol': 1e-14}), method='2s' )
print(results3)


print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
 
labels = ['x', 'satellite', 'wired', 'prices']
true_beta = [1, 4, 4, -2]
 
print(f"\n{'Param':>18} {'Demand Only':>16} {'D+S BLP':>16} {'D+S Optimal':>16} {'True':>8}")
print("-" * 78)
for i, (lab, tv) in enumerate(zip(labels, true_beta)):
    print(f"{lab:>18} {results1.beta[i,0]:>8.4f}({results1.beta_se[i,0]:.3f}) "
          f"{results2.beta[i,0]:>8.4f}({results2.beta_se[i,0]:.3f}) "
          f"{results3.beta[i,0]:>8.4f}({results3.beta_se[i,0]:.3f}) {tv:>8}")
 
print(f"{'sigma_sat':>18} {results1.sigma[0,0]:>8.4f}({results1.sigma_se[0,0]:.3f}) "
      f"{results2.sigma[0,0]:>8.4f}({results2.sigma_se[0,0]:.3f}) "
      f"{results3.sigma[0,0]:>8.4f}({results3.sigma_se[0,0]:.3f}) {'1':>8}")
print(f"{'sigma_wired':>18} {results1.sigma[1,1]:>8.4f}({results1.sigma_se[1,1]:.3f}) "
      f"{results2.sigma[1,1]:>8.4f}({results2.sigma_se[1,1]:.3f}) "
      f"{results3.sigma[1,1]:>8.4f}({results3.sigma_se[1,1]:.3f}) {'1':>8}")
 
print(f"\n{'Supply':>18} {'D+S BLP':>16} {'D+S Optimal':>16} {'True':>8}")
print("-" * 60)
supply_labels = ['gamma0 (const)', 'gamma1 (w)']
true_gamma = [0.5, 0.25]
for i, (lab, tv) in enumerate(zip(supply_labels, true_gamma)):
    print(f"{lab:>18} {results2.gamma[i,0]:>8.4f}({results2.gamma_se[i,0]:.3f}) "
          f"{results3.gamma[i,0]:>8.4f}({results3.gamma_se[i,0]:.3f}) {tv:>8}")


SPEC 1: Demand only, BLP instruments
Dimensions:
 T    N     F     I     K1    K2    MD 
---  ----  ---  -----  ----  ----  ----
600  2400   4   29400   4     2     8  

Formulations:
       Column Indices:             0          1        2      3   
-----------------------------  ---------  ---------  -----  ------
 X1: Linear Characteristics        x      satellite  wired  prices
X2: Nonlinear Characteristics  satellite    wired                 
Problem Results Summary:
GMM   Objective     Projected    Reduced Hessian  Reduced Hessian  Clipped  Weighting Matrix  Covariance Matrix
Step    Value     Gradient Norm  Min Eigenvalue   Max Eigenvalue   Shares   Condition Number  Condition Number 
----  ----------  -------------  ---------------  ---------------  -------  ----------------  -----------------
 2    +1.279E+00   +5.605E-07      +1.746E-01       +1.145E+00        0        +2.108E+02        +5.625E+04    

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   

Question 6

In [15]:
T = 600
J = 4

elasticities = results2.compute_elasticities()
diversion_ratios = results2.compute_diversion_ratios()

avg_elast = elasticities.reshape((T, J, J)).mean(axis=0)
avg_div = diversion_ratios.reshape((T, J, J)).mean(axis=0)

true_elast = pd.read_csv('true_ownprice_elasticities.csv', index_col=0)
true_div = pd.read_csv('true_diversionratio.csv', index_col=0)

print("OWN-PRICE ELASTICITIES")
print(f"{'Product':<10} {'Estimated':<15} {'True':<15}")
for j in range(J):
    print(f"J{j+1:<9} {avg_elast[j,j]:<15.4f} {true_elast.iloc[0, j]:<15.4f}")

print("\nESTIMATED DIVERSION RATIOS")
print(pd.DataFrame(avg_div, index=['J1','J2','J3','J4'], columns=['J1','J2','J3','J4']).round(4))

print("\nTRUE DIVERSION RATIOS")
print(true_div.round(4))

OWN-PRICE ELASTICITIES
Product    Estimated       True           
J1         -5.5076         -4.2500        
J2         -5.6922         -4.3714        
J3         -5.4323         -4.2711        
J4         -5.4256         -4.2576        

ESTIMATED DIVERSION RATIOS
        J1      J2      J3      J4
J1  0.3646  0.2732  0.1775  0.1847
J2  0.2773  0.3682  0.1770  0.1775
J3  0.1852  0.1840  0.3254  0.3054
J4  0.1882  0.1790  0.3038  0.3290

TRUE DIVERSION RATIOS
        J1      J2      J3      J4
J1  0.3338  0.2170  0.2204  0.2288
J2  0.2194  0.3370  0.2216  0.2220
J3  0.2193  0.2185  0.3350  0.2272
J4  0.2246  0.2150  0.2232  0.3371


Question 9

In [14]:
product_data['merger_ids'] = product_data['firm_ids'].replace({1: 0})

merger_prices_12 = results2.compute_prices(firm_ids=product_data['merger_ids'])

price_changes_12 = merger_prices_12 - product_data['prices'].values.reshape(-1, 1)

T = 600
J = 4
avg_price_change_12 = price_changes_12.reshape((T, J)).mean(axis=0)
avg_pct_change_12 = ((merger_prices_12.reshape((T, J)) / product_data['prices'].values.reshape((T, J))) - 1).mean(axis=0) * 100

print("MERGER 1+2 (satellite + satellite)")
print(f"{'Product':<10} {'Avg Price Change':<20} {'Avg % Change':<15}")
for j in range(J):
    print(f"J{j+1:<9} {avg_price_change_12[j]:<20.4f} {avg_pct_change_12[j]:<15.2f}%")

MERGER 1+2 (satellite + satellite)
Product    Avg Price Change     Avg % Change   
J1         0.1822               7.46           %
J2         0.1842               7.21           %
J3         0.0062               0.27           %
J4         0.0049               0.20           %


Question 10

In [13]:
product_data['merger_ids_13'] = product_data['firm_ids'].replace({2: 0})

merger_prices_13 = results2.compute_prices(firm_ids=product_data['merger_ids_13'])

T = 600
J = 4
avg_pct_change_13 = ((merger_prices_13.reshape((T, J)) / product_data['prices'].values.reshape((T, J))) - 1).mean(axis=0) * 100

print("MERGER 1+3 (satellite + wired)")
print(f"{'Product':<10} {'Avg % Change':<15}")
for j in range(J):
    print(f"J{j+1:<9} {avg_pct_change_13[j]:<15.2f}%")

print("\nCOMPARISON")
print(f"{'Product':<10} {'Merger 1+2':<15} {'Merger 1+3':<15}")
for j in range(J):
    print(f"J{j+1:<9} {avg_pct_change_12[j]:<15.2f}% {avg_pct_change_13[j]:<15.2f}%")

MERGER 1+3 (satellite + wired)
Product    Avg % Change   
J1         4.55           %
J2         0.23           %
J3         4.57           %
J4         0.25           %

COMPARISON
Product    Merger 1+2      Merger 1+3     
J1         7.46           % 4.55           %
J2         7.21           % 0.23           %
J3         0.27           % 4.57           %
J4         0.20           % 0.25           %


Question 12

In [12]:
costs = results2.compute_costs()

merger_costs = costs.copy()
merger_costs[product_data['firm_ids'].isin([0, 1])] = 0.85 * merger_costs[product_data['firm_ids'].isin([0, 1])]

merger_prices_eff = results2.compute_prices(firm_ids=product_data['merger_ids'], costs=merger_costs)

T = 600
J = 4
avg_pct_change_eff = ((merger_prices_eff.reshape((T, J)) / product_data['prices'].values.reshape((T, J))) - 1).mean(axis=0) * 100

print("MERGER 1+2 WITH 15% COST REDUCTION")
print(f"{'Product':<10} {'No Savings':<15} {'15% Savings':<15}")
for j in range(J): 
    print(f"J{j+1:<9} {avg_pct_change_12[j]:<15.2f}% {avg_pct_change_eff[j]:<15.2f}%")

cs_no_merger = results2.compute_consumer_surpluses()
cs_post_merger = results2.compute_consumer_surpluses(prices=merger_prices_eff)

cs_change = cs_post_merger - cs_no_merger
avg_cs_change = cs_change.mean()
total_cs_change = cs_change.sum()

print(f"\nAverage CS change per market: {avg_cs_change:.4f}")
print(f"Total CS change across all markets: {total_cs_change:.4f}")

MERGER 1+2 WITH 15% COST REDUCTION
Product    No Savings      15% Savings    
J1         7.46           % -0.07          %
J2         7.21           % -0.59          %
J3         0.27           % -0.22          %
J4         0.20           % -0.29          %

Average CS change per market: 0.0207
Total CS change across all markets: 12.4258
